# 02 · Silver transform
Each cleaning function in `food_pipeline/silver.py` fixes one issue found in `00_explore.ipynb`.
This notebook applies them one at a time on Bronze, so you can see what each one changes.

Step 1 only cleans in memory. Data quality rules, deduplication and writing the Silver table come in the next steps.

In [1]:
from pathlib import Path
from pyspark.sql import functions as F
from food_pipeline.spark_session import get_spark
from food_pipeline import silver

ROOT = Path.cwd().parent
spark = get_spark()
BRONZE = (ROOT / "lakehouse" / "bronze" / "survey_raw").as_posix()

bronze = spark.read.format("delta").load(BRONZE)
print("bronze rows:", bronze.count())

bronze rows: 388


## 1 · `trim_strings`
Fixes `"Negative "` (trailing space). Brackets make spaces visible: before vs after.

In [2]:
def bracketed(df, column):
    return df.groupBy(F.concat(F.lit("["), F.col(column), F.lit("]")).alias(column)).count()

trimmed = silver.trim_strings(bronze)
bracketed(bronze, "feedback").show()
bracketed(trimmed, "feedback").show()

+-----------+-----+
|   feedback|count|
+-----------+-----+
|[Negative ]|   71|
| [Positive]|  317|
+-----------+-----+



+----------+-----+
|  feedback|count|
+----------+-----+
|[Positive]|  317|
|[Negative]|   71|
+----------+-----+



## 2 · `standardise_categories`
`Self Employeed` → `Self Employed`, `House wife` → `Homemaker`.

In [3]:
silver.standardise_categories(trimmed).groupBy("occupation").count().orderBy(F.desc("count")).show()

+-------------+-----+
|   occupation|count|
+-------------+-----+
|      Student|  207|
|     Employee|  118|
|Self Employed|   54|
|    Homemaker|    9|
+-------------+-----+



## 3 · `cast_types`
`age`, `family_size` → int; `latitude`, `longitude` → double; `pin_code` stays a string (it's an ID).
A value that can't be cast becomes **null**, so we count nulls to prove nothing was lost.

In [4]:
typed = silver.cast_types(trimmed)
typed.select("age", "family_size", "latitude", "longitude", "pin_code").printSchema()
typed.select([F.sum(F.col(c).isNull().cast("int")).alias(f"{c}_nulls")
              for c in silver.COLUMN_TYPES]).show()
typed.select(F.min("age"), F.max("age"), F.min("family_size"), F.max("family_size")).show()

root
 |-- age: integer (nullable = true)
 |-- family_size: integer (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- pin_code: string (nullable = true)



+---------+-----------------+--------------+---------------+
|age_nulls|family_size_nulls|latitude_nulls|longitude_nulls|
+---------+-----------------+--------------+---------------+
|        0|                0|             0|              0|
+---------+-----------------+--------------+---------------+



+--------+--------+----------------+----------------+
|min(age)|max(age)|min(family_size)|max(family_size)|
+--------+--------+----------------+----------------+
|      18|      33|               1|               6|
+--------+--------+----------------+----------------+



## 4 · `map_income`
Text bands get a sortable number plus numeric bounds. An unknown band would map to null.

In [5]:
(silver.map_income(trimmed)
    .groupBy("monthly_income", "income_band_order", "income_min_inr", "income_max_inr").count()
    .orderBy("income_band_order").show(truncate=False))

+---------------+-----------------+--------------+--------------+-----+
|monthly_income |income_band_order|income_min_inr|income_max_inr|count|
+---------------+-----------------+--------------+--------------+-----+
|No Income      |0                |0             |0             |187  |
|Below Rs.10000 |1                |1             |10000         |25   |
|10001 to 25000 |2                |10001         |25000         |45   |
|25001 to 50000 |3                |25001         |50000         |69   |
|More than 50000|4                |50001         |NULL          |62   |
+---------------+-----------------+--------------+--------------+-----+



## 5 · `clean_outcome`
`output` (Yes/No text) → `ordered_again` (boolean). Assumption, documented in the README: `Output` means *the respondent would order again*.

In [6]:
silver.clean_outcome(trimmed).groupBy("ordered_again").count().show()

+-------------+-----+
|ordered_again|count|
+-------------+-----+
|         true|  301|
|        false|   87|
+-------------+-----+



## All steps together: `clean`
Row count must not change: cleaning fixes values, it never adds or drops rows.

In [7]:
cleaned = silver.clean(bronze)
print("rows in:", bronze.count(), "rows out:", cleaned.count())
cleaned.printSchema()
cleaned.drop("_ingest_ts", "_source_file").show(5, truncate=False)

rows in: 388 rows out: 388
root
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- monthly_income: string (nullable = true)
 |-- educational_qualifications: string (nullable = true)
 |-- family_size: integer (nullable = true)
 |-- customer_type: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- pin_code: string (nullable = true)
 |-- feedback: string (nullable = true)
 |-- raw_extra: string (nullable = true)
 |-- _ingest_ts: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _batch_id: string (nullable = true)
 |-- income_band_order: integer (nullable = true)
 |-- income_min_inr: integer (nullable = true)
 |-- income_max_inr: integer (nullable = true)
 |-- ordered_again: boolean (nullable = true)



+---+------+--------------+----------+--------------+--------------------------+-----------+-------------+--------+---------+--------+--------+---------+---------+-----------------+--------------+--------------+-------------+
|age|gender|marital_status|occupation|monthly_income|educational_qualifications|family_size|customer_type|latitude|longitude|pin_code|feedback|raw_extra|_batch_id|income_band_order|income_min_inr|income_max_inr|ordered_again|
+---+------+--------------+----------+--------------+--------------------------+-----------+-------------+--------+---------+--------+--------+---------+---------+-----------------+--------------+--------------+-------------+
|20 |Female|Single        |Student   |No Income     |Post Graduate             |4          |Frequent     |12.9766 |77.5993  |560001  |Positive|Yes      |batch_01 |0                |0             |0             |true         |
|24 |Female|Single        |Student   |Below Rs.10000|Graduate                  |3          |Regu